# Parallel Audio Transcription

This notebook implements parallel transcription by splitting audio into 2 chunks and processing them simultaneously:
- Chunk 1: Processed on CPU
- Chunk 2: Processed on CoreML/GPU (Apple Neural Engine)

## Setup Requirements

### Option 1: MLX Whisper (Recommended for Mac M4)
```bash
pip install mlx-whisper
```

### Option 2: whisper.cpp with CoreML
```bash
brew install whisper.cpp
pip install pywhispercpp
```

### Option 3: Parallel faster-whisper (Fallback)
Use existing faster-whisper with multiple CPU processes


In [7]:
#%pip install mlx-whisper
#%pip install pywhispercpp

In [8]:
import os
import subprocess
import time
from multiprocessing import Process, Queue
from pathlib import Path
import json


In [9]:
# Configuration
input_audio_path = '/Users/vlad/Library/CloudStorage/GoogleDrive-vladimir.smirnov@rhodaris.com/Shared drives/rhodaris working drive/work/JobSearch/EPAM 2026 01 20 AI PM/2.10.2026, 1.27.35 PM.webm'
output_audio_name = "2nd interview 2026 02 10_output16khz.wav"
output_transcription_name = "2nd interview 2026 02 10_transcription_parallel.txt"
meeting_language = "en"

# Parallel processing settings
OVERLAP_SECONDS = 2  # Overlap between chunks to avoid cutting words
MODEL_SIZE = "large-v2"  # Options: tiny, base, small, medium, large-v2, large-v3
GPU_RATIO = 0.78  # Ratio for GPU: 0.78 = 78% to GPU, 22% to CPU (based on speed difference)

# Setup paths
folder_path = os.path.dirname(input_audio_path)
output_audio_path = os.path.join(folder_path, output_audio_name)
output_file_path = os.path.join(folder_path, output_transcription_name)
temp_dir = os.path.join(folder_path, "temp_chunks")

# Create temp directory for chunks
os.makedirs(temp_dir, exist_ok=True)

print(f"Temp directory: {temp_dir}")


Temp directory: /Users/vlad/Library/CloudStorage/GoogleDrive-vladimir.smirnov@rhodaris.com/Shared drives/rhodaris working drive/work/JobSearch/EPAM 2026 01 20 AI PM/temp_chunks


In [10]:
# Detect available transcription backends
def detect_available_backends():
    """Detect which transcription backends are available on this system."""
    backends = {}
    
    # Check for MLX Whisper (best for Mac)
    try:
        import mlx_whisper
        backends['mlx'] = True
        print("✓ MLX Whisper available (optimal for Mac M4)")
    except ImportError:
        backends['mlx'] = False
        print("✗ MLX Whisper not available")
    
    # Check for whisper.cpp
    try:
        import pywhispercpp
        backends['whispercpp'] = True
        print("✓ whisper.cpp available")
    except ImportError:
        backends['whispercpp'] = False
        print("✗ whisper.cpp not available")
    
    # Check for faster-whisper (already installed)
    try:
        from faster_whisper import WhisperModel
        backends['faster_whisper'] = True
        print("✓ faster-whisper available")
    except ImportError:
        backends['faster_whisper'] = False
        print("✗ faster-whisper not available")
    
    return backends

available_backends = detect_available_backends()
print(f"\nAvailable backends: {[k for k, v in available_backends.items() if v]}")


✓ MLX Whisper available (optimal for Mac M4)
✓ whisper.cpp available
✓ faster-whisper available

Available backends: ['mlx', 'whispercpp', 'faster_whisper']


In [11]:
# Preprocess audio with ffmpeg (16kHz, mono)
print("Preprocessing audio...")
start_preprocess = time.time()

subprocess.run([
    "ffmpeg", "-y", "-vn", "-i", input_audio_path, 
    "-ar", "16000", "-ac", "1", output_audio_path
], check=True, capture_output=True)

preprocess_time = time.time() - start_preprocess
print(f"Audio preprocessed in {preprocess_time:.2f}s: {output_audio_path}")

# Get audio duration
result = subprocess.run([
    "ffprobe", "-v", "error", "-show_entries",
    "format=duration", "-of",
    "default=noprint_wrappers=1:nokey=1", output_audio_path
], capture_output=True, text=True)

audio_duration = float(result.stdout.strip())
print(f"Audio duration: {audio_duration:.2f}s ({audio_duration/60:.2f} minutes)")


Preprocessing audio...
Audio preprocessed in 4.41s: /Users/vlad/Library/CloudStorage/GoogleDrive-vladimir.smirnov@rhodaris.com/Shared drives/rhodaris working drive/work/JobSearch/EPAM 2026 01 20 AI PM/2nd interview 2026 02 10_output16khz.wav
Audio duration: 3762.78s (62.71 minutes)


In [12]:
# Split audio into 2 chunks with overlap
def split_audio_with_overlap(input_path, output_dir, overlap_seconds=2, gpu_ratio=0.78):
    """
    Split audio into 2 chunks with overlap, optimized for different processing speeds.
    
    Args:
        input_path: Path to preprocessed audio file
        output_dir: Directory to save chunks
        overlap_seconds: Seconds of overlap between chunks
        gpu_ratio: Ratio of audio to assign to GPU (0.78 = 78% to GPU, 22% to CPU)
    
    Returns:
        List of (chunk_path, start_time, end_time) tuples
    """
    # Get audio duration
    result = subprocess.run([
        "ffprobe", "-v", "error", "-show_entries",
        "format=duration", "-of",
        "default=noprint_wrappers=1:nokey=1", input_path
    ], capture_output=True, text=True, check=True)
    
    duration = float(result.stdout.strip())
    # Split based on processing speed ratio (CPU gets smaller chunk)
    split_point = duration * (1 - gpu_ratio)
    
    chunks = []
    
    # Chunk 1: 0 to (split_point + overlap)
    chunk1_path = os.path.join(output_dir, "chunk_1.wav")
    chunk1_duration = split_point + overlap_seconds
    subprocess.run([
        "ffmpeg", "-y", "-i", input_path,
        "-t", str(chunk1_duration),
        "-c", "copy", chunk1_path
    ], check=True, capture_output=True)
    chunks.append((chunk1_path, 0, split_point + overlap_seconds))
    print(f"Chunk 1 (CPU): 0s to {chunk1_duration:.2f}s ({chunk1_duration/60:.2f} min)")
    
    # Chunk 2: (split_point - overlap) to end
    chunk2_path = os.path.join(output_dir, "chunk_2.wav")
    chunk2_start = max(0, split_point - overlap_seconds)
    subprocess.run([
        "ffmpeg", "-y", "-i", input_path,
        "-ss", str(chunk2_start),
        "-c", "copy", chunk2_path
    ], check=True, capture_output=True)
    chunks.append((chunk2_path, chunk2_start, duration))
    chunk2_duration = duration - chunk2_start
    print(f"Chunk 2 (GPU): {chunk2_start:.2f}s to {duration:.2f}s ({chunk2_duration/60:.2f} min)")
    print(f"\nSplit ratio: {(1-gpu_ratio)*100:.0f}% CPU / {gpu_ratio*100:.0f}% GPU")
    
    return chunks, split_point

print("Splitting audio into 2 chunks with overlap (optimized for GPU/CPU speed)...")
start_split = time.time()
chunks, split_point = split_audio_with_overlap(output_audio_path, temp_dir, OVERLAP_SECONDS, GPU_RATIO)
split_time = time.time() - start_split
print(f"Audio split completed in {split_time:.2f}s")
print(f"Split point: {split_point:.2f}s ({split_point/60:.2f} minutes)")


Splitting audio into 2 chunks with overlap (optimized for GPU/CPU speed)...
Chunk 1 (CPU): 0s to 829.81s (13.83 min)
Chunk 2 (GPU): 825.81s to 3762.78s (48.95 min)

Split ratio: 22% CPU / 78% GPU
Audio split completed in 0.20s
Split point: 827.81s (13.80 minutes)


In [13]:
# Import transcription worker functions from separate module
# (This avoids multiprocessing pickling issues in Jupyter notebooks)
from transcribe_workers import (
    transcribe_with_mlx,
    transcribe_with_faster_whisper,
    transcribe_with_whispercpp
)

print("Transcription functions loaded from transcribe_workers.py")


Transcription functions loaded from transcribe_workers.py


In [14]:
# Parallel transcription manager
def transcribe_parallel(chunks_info, language, backends):
    """
    Transcribe chunks in parallel using different backends.
    
    Strategy:
    - Chunk 1: CPU-based (faster-whisper)
    - Chunk 2: GPU/CoreML-based (MLX or whisper.cpp with CoreML)
    """
    result_queue = Queue()
    processes = []
    
    print("\n" + "="*60)
    print("Starting parallel transcription...")
    print("="*60)
    
    # Chunk 1: CPU process
    chunk1_path, _, _ = chunks_info[0]
    if backends.get('faster_whisper'):
        p1 = Process(target=transcribe_with_faster_whisper, 
                     args=(chunk1_path, language, result_queue, 1, MODEL_SIZE, "cpu", "int8"))
        processes.append(p1)
        print("Process 1: faster-whisper (CPU)")
    else:
        print("ERROR: No CPU backend available!")
        return None
    
    # Chunk 2: CoreML/GPU process
    chunk2_path, _, _ = chunks_info[1]
    if backends.get('mlx'):
        p2 = Process(target=transcribe_with_mlx,
                     args=(chunk2_path, language, result_queue, 2, MODEL_SIZE))
        processes.append(p2)
        print("Process 2: MLX Whisper (CoreML/Neural Engine)")
    elif backends.get('whispercpp'):
        p2 = Process(target=transcribe_with_whispercpp,
                     args=(chunk2_path, language, result_queue, 2, MODEL_SIZE, True))
        processes.append(p2)
        print("Process 2: whisper.cpp (CoreML)")
    else:
        # Fallback: Use faster-whisper on CPU with different threads
        p2 = Process(target=transcribe_with_faster_whisper,
                     args=(chunk2_path, language, result_queue, 2, MODEL_SIZE, "cpu", "int8"))
        processes.append(p2)
        print("Process 2: faster-whisper (CPU) - FALLBACK")
        print("⚠️  No CoreML backend available, using CPU for both chunks")
    
    # Start all processes
    start_time = time.time()
    for p in processes:
        p.start()
    
    # Collect results as they come (prevents queue blocking)
    results = []
    num_expected = len(processes)
    
    while len(results) < num_expected:
        try:
            result = result_queue.get(timeout=10)  # 10 second timeout
            results.append(result)
        except:
            # Check if processes are still alive
            if not any(p.is_alive() for p in processes):
                break  # All processes done, exit loop
    
    # Now safely join processes
    for p in processes:
        p.join(timeout=5)  # Add timeout to prevent infinite wait
        if p.is_alive():
            print(f"Warning: Process {p.pid} didn't terminate, forcing...")
            p.terminate()
            p.join(timeout=2)
    
    total_time = time.time() - start_time
    
    # Sort by chunk_id
    results.sort(key=lambda x: x.get('chunk_id', 0))
    
    print("\n" + "="*60)
    print(f"Parallel transcription completed in {total_time:.2f}s")
    for r in results:
        if 'error' not in r:
            print(f"  Chunk {r['chunk_id']}: {r['time']:.2f}s ({r['backend']})")
        else:
            print(f"  Chunk {r['chunk_id']}: ERROR - {r['error']}")
    print("="*60 + "\n")
    
    return results, total_time

# Run parallel transcription
results, parallel_time = transcribe_parallel(chunks, meeting_language, available_backends)



Starting parallel transcription...
Process 1: faster-whisper (CPU)
Process 2: MLX Whisper (CoreML/Neural Engine)
[Chunk 2] Starting MLX transcription (CoreML/Neural Engine)...
[Chunk 1] Starting faster-whisper transcription (cpu)...


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 51463.85it/s]


[Chunk 2] MLX transcription completed in 187.51s
[Chunk 1] faster-whisper transcription completed in 643.68s

Parallel transcription completed in 645.09s
  Chunk 1: 643.68s (faster-whisper-cpu)
  Chunk 2: 187.51s (mlx)



In [15]:
# Merge transcription results with overlap handling
def merge_transcriptions(results, chunks_info, split_point, overlap_seconds):
    """
    Merge transcription results from multiple chunks.
    Handle overlap region by keeping only the first chunk's text in overlap.
    
    Args:
        results: List of result dictionaries from parallel transcription
        chunks_info: List of (chunk_path, start_time, end_time) tuples
        split_point: Time in seconds where audio was split
        overlap_seconds: Seconds of overlap between chunks
    
    Returns:
        Complete transcription text
    """
    if any('error' in r for r in results):
        print("ERROR: Some chunks failed to transcribe")
        return None, []
    
    merged_segments = []
    
    # Process chunk 1 (keep all segments)
    chunk1_result = [r for r in results if r['chunk_id'] == 1][0]
    chunk1_start_offset = chunks_info[0][1]  # Start time of chunk 1
    
    for segment in chunk1_result['segments']:
        abs_start = segment['start'] + chunk1_start_offset
        abs_end = segment['end'] + chunk1_start_offset
        
        # Only include segments before split point (exclude overlap)
        if abs_start < split_point:
            merged_segments.append({
                'start': abs_start,
                'end': abs_end,
                'text': segment['text']
            })
    
    # Process chunk 2 (skip overlap region)
    chunk2_result = [r for r in results if r['chunk_id'] == 2][0]
    chunk2_start_offset = chunks_info[1][1]  # Start time of chunk 2
    
    for segment in chunk2_result['segments']:
        abs_start = segment['start'] + chunk2_start_offset
        abs_end = segment['end'] + chunk2_start_offset
        
        # Only include segments after split point (skip overlap)
        if abs_start >= split_point:
            merged_segments.append({
                'start': abs_start,
                'end': abs_end,
                'text': segment['text']
            })
    
    # Sort by start time (should already be sorted, but just in case)
    merged_segments.sort(key=lambda x: x['start'])
    
    # Combine all text
    transcription = " ".join([seg['text'].strip() for seg in merged_segments])
    
    print(f"Merged {len(merged_segments)} segments")
    print(f"  Chunk 1 contributed: {len([s for s in merged_segments if s['start'] < split_point])} segments")
    print(f"  Chunk 2 contributed: {len([s for s in merged_segments if s['start'] >= split_point])} segments")
    
    return transcription, merged_segments

# Merge results
print("\nMerging transcription results...")
transcription, merged_segments = merge_transcriptions(results, chunks, split_point, OVERLAP_SECONDS)

if transcription:
    print(f"✓ Transcription complete: {len(transcription)} characters")
else:
    print("✗ Transcription failed")



Merging transcription results...
Merged 371 segments
  Chunk 1 contributed: 168 segments
  Chunk 2 contributed: 203 segments
✓ Transcription complete: 40813 characters


In [16]:
# Save transcription
if transcription:
    with open(output_file_path, "w", encoding="utf-8") as file:
        file.write(transcription)
    print(f"\n✓ Transcription saved to: {output_file_path}")
    
    # Also save detailed segments as JSON
    segments_path = output_file_path.replace(".txt", "_segments.json")
    with open(segments_path, "w", encoding="utf-8") as file:
        json.dump(merged_segments, file, indent=2, ensure_ascii=False)
    print(f"✓ Segments saved to: {segments_path}")
else:
    print("\n✗ No transcription to save")



✓ Transcription saved to: /Users/vlad/Library/CloudStorage/GoogleDrive-vladimir.smirnov@rhodaris.com/Shared drives/rhodaris working drive/work/JobSearch/EPAM 2026 01 20 AI PM/2nd interview 2026 02 10_transcription_parallel.txt
✓ Segments saved to: /Users/vlad/Library/CloudStorage/GoogleDrive-vladimir.smirnov@rhodaris.com/Shared drives/rhodaris working drive/work/JobSearch/EPAM 2026 01 20 AI PM/2nd interview 2026 02 10_transcription_parallel_segments.json


In [17]:
# Cleanup temporary files
import shutil
cleanup = 'y'
#cleanup = input("Delete temporary chunk files? (y/n): ").lower().strip()
if cleanup == 'y':
    try:
        shutil.rmtree(temp_dir)
        print(f"✓ Cleaned up temporary directory: {temp_dir}")
    except Exception as e:
        print(f"✗ Failed to cleanup: {e}")
else:
    print(f"Temporary files kept in: {temp_dir}")


✓ Cleaned up temporary directory: /Users/vlad/Library/CloudStorage/GoogleDrive-vladimir.smirnov@rhodaris.com/Shared drives/rhodaris working drive/work/JobSearch/EPAM 2026 01 20 AI PM/temp_chunks


In [18]:
# Performance Summary
print("\n" + "="*60)
print("PERFORMANCE SUMMARY")
print("="*60)
print(f"Audio duration: {audio_duration:.2f}s ({audio_duration/60:.2f} minutes)")
print(f"\nPreprocessing: {preprocess_time:.2f}s")
print(f"Audio splitting: {split_time:.2f}s")
print(f"Parallel transcription: {parallel_time:.2f}s")
print(f"\nTotal processing time: {preprocess_time + split_time + parallel_time:.2f}s")
print(f"Real-time factor: {(preprocess_time + split_time + parallel_time) / audio_duration:.2f}x")
print(f"  (Lower is better - 0.5x means 2x faster than real-time)")
print("\n" + "="*60)

# Compare with original method
print("\nCOMPARISON WITH ORIGINAL METHOD:")
print("To compare with the original single-process transcription,")
print("run the transcribe.ipynb notebook and compare times.")
print("\nExpected speedup factors:")
print("  - If using MLX: 2-4x faster overall")
print("  - Parallel processing: Additional 1.5-2x speedup")
print("  - Combined: 3-8x faster than original faster-whisper")



PERFORMANCE SUMMARY
Audio duration: 3762.78s (62.71 minutes)

Preprocessing: 4.41s
Audio splitting: 0.20s
Parallel transcription: 645.09s

Total processing time: 649.69s
Real-time factor: 0.17x
  (Lower is better - 0.5x means 2x faster than real-time)


COMPARISON WITH ORIGINAL METHOD:
To compare with the original single-process transcription,
run the transcribe.ipynb notebook and compare times.

Expected speedup factors:
  - If using MLX: 2-4x faster overall
  - Parallel processing: Additional 1.5-2x speedup
  - Combined: 3-8x faster than original faster-whisper


## Notes

### Backend Selection Priority
1. **MLX Whisper** (Recommended) - Best performance on Mac M4, uses Neural Engine
2. **whisper.cpp with CoreML** - Good performance, requires manual installation
3. **faster-whisper parallel** - Fallback option, uses multiple CPU cores

### Installation Instructions

#### For MLX Whisper (Easiest and Fastest):
```bash
pip install mlx-whisper
```

#### For whisper.cpp with CoreML:
```bash
# Install whisper.cpp
brew install whisper.cpp

# Download CoreML models
mkdir -p ~/.whisper-cpp-models
cd ~/.whisper-cpp-models
# Download from: https://huggingface.co/ggerganov/whisper.cpp

# Install Python bindings
pip install pywhispercpp
```

### Performance Tips
- Use `medium` model for better speed (2-3x faster) with good accuracy
- Use `large-v2` for best accuracy but slower processing
- Increase `OVERLAP_SECONDS` to 3-4 for better word boundary handling
- Monitor CPU/GPU usage during transcription to verify parallel execution

### Troubleshooting
- If MLX fails: Check that you're using Python 3.10+ and Apple Silicon Mac
- If CoreML fails: Ensure Xcode Command Line Tools are installed: `xcode-select --install`
- If both processes use CPU: Check available backends cell output
- If accuracy is poor at split point: Increase OVERLAP_SECONDS
